In [1]:
# import libary
from datasets import load_dataset

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# laod the dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations")

In [3]:
# overview of the dataset
dataset

DatasetDict({
    train: Dataset({
        features: ['Context', 'Response'],
        num_rows: 3512
    })
})

In [4]:
# One sample
dataset["train"][0]

{'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is someh

In [5]:
# rename colums with meaningful keys
dataset = dataset["train"].rename_columns({"Context":"query","Response":"response"})

In [6]:
# One sample
dataset[0]

{'query': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
 'response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow

In [7]:
# remove empty queries and responses
dataset = dataset.filter(lambda sample: sample["query"] is not None and sample["query"].strip() != "")
dataset = dataset.filter(lambda sample: sample["response"] is not None and sample["response"].strip() != "")


In [8]:
# compute length
def compute_length(sample, new_key, field_key):
    return {new_key: len(sample[field_key].split(" "))}

In [9]:
# Adding length property for both queries and respnses
dataset = dataset.map(lambda sample: compute_length(sample, new_key="query_length", field_key="query"))
dataset = dataset.map(lambda sample: compute_length(sample, new_key="response_length", field_key="response"))

In [10]:
dataset[0]

{'query': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
 'response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow

In [10]:
# removing short response
dataset = dataset.filter(lambda x:x["response_length"] > 30)
len(dataset)

3437

In [12]:
# clean new line and extra spacing from the query and response
import re

def clean_text(text):
    """Clean extra whitespace, newlines, and normalize spacing"""
    if not text:
        return text
    
    # Replace multiple whitespace (including newlines, tabs) with single space
    text = re.sub(r'\s+', ' ', text)
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    return text

def clean_dataset_text(example):
    """Clean query and response text"""
    return {
        "query": clean_text(example["query"]),
        "response": clean_text(example["response"])
    }

In [13]:
dataset = dataset.map(clean_dataset_text)

Map: 100%|██████████| 3437/3437 [00:00<00:00, 3942.54 examples/s]


In [14]:
dataset[0]

{'query': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?",
 'response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.Ba

In [15]:
from transformers import AutoTokenizer
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [16]:
def prepare_dataset_with_text_column(dataset):
   
    
    # Data prompt
    data_prompt = """
### Input:
Pateint Query: {}

### Response:
{}"""

    # Formatting the prompt
    EOS_TOKEN = tokenizer.eos_token
    def formatting_prompt(examples):
        # sys_prompts = examples["instruction"]
        queries = examples["query"]
        responses = examples["response"]
        texts = []
        
        for query, response in zip (queries, responses):
            text = data_prompt.format(query, response) + EOS_TOKEN
            texts.append(text)
        
        return {"text": texts}
    
    dataset_with_text_column = dataset.map(formatting_prompt, batched=True)
    
    return dataset_with_text_column



In [17]:
dataset = prepare_dataset_with_text_column(dataset)

Map: 100%|██████████| 3437/3437 [00:00<00:00, 57448.21 examples/s]


In [18]:
print(dataset[50]["text"])


### Input:
Pateint Query: I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac. I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years. I’ve never had counseling about any of this. Do I have too many issues to address in counseling?

### Response:
Usually people who struggle with depression also have symptoms of anxiety. These symptoms come about as a result of underlying causes. The obstacles you have been overcoming such as abuse, cancer and insomnia likely bring about similar feelings and emotions in you which affect your self-esteem. Counseling is meant to support people who have layers of difficulties. We see the difficulties as layers to an onion, peeling away the many layers.<|eot_id|>


In [19]:
dataset_full = dataset.shuffle(seed=42).train_test_split(train_size=0.8)

In [20]:
dataset_full

DatasetDict({
    train: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 2749
    })
    test: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 688
    })
})

In [21]:
dataset_train_validation = dataset_full["test"]
dataset_train_validation = dataset_train_validation.train_test_split(test_size=0.5, seed=42)

In [22]:
dataset_train_validation

DatasetDict({
    train: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 344
    })
    test: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 344
    })
})

In [23]:
dataset_full["test"] = dataset_train_validation["train"]
dataset_full["validation"] = dataset_train_validation["test"]

In [24]:
dataset_full

DatasetDict({
    train: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 2749
    })
    test: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 344
    })
    validation: Dataset({
        features: ['query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 344
    })
})

In [25]:
# Save to disk
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
dataset_full.save_to_disk(f"{dataset_based_path}/mh-amod-conversation-v2")

Saving the dataset (1/1 shards): 100%|██████████| 344/344 [00:00<00:00, 1202.59 examples/s]


In [2]:
from datasets import load_from_disk
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset/mh-amod-conversation-v2"
dataset = load_from_disk(dataset_based_path)

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
import textwrap

sample = 150
print(dataset["train"][sample])

# Wrap text at 80 characters per line
print("Response\n")
print("\n".join(textwrap.wrap(dataset["train"][sample]["response"], width=100)))
# print("=" * 80)


{'query': "My boyfriend of five months expresses how much he cares for me and loves me, and I believe it. It has been one of the best relationships I have been in for a short period of time, but he has lied to me so much. He’ll tell me that he'll stop lying, and while he’s saying that, he's hiding something or lying about something. First, it was about a girl. He hid text messages and claimed she was a friend and it was a friendly conversation when it was a girl he once found interest in. Next was his ex-girlfriend. He put a fake contact name, and it was just friendly, but he lied to my face. All the time, he claims he never cheated and still expresses his love the same way and that he wants to change and do better.", 'response': "New York, New York...it's not your boyfriend's lying that concerns me the most; it's the lies you tell yourself. Verbal expressions are easy, but it's walking the walk that's more important in the end. When someone shows you what they're capable of, you can b